In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions

# Configure accelerator options for GPU
accelerator_options = AcceleratorOptions(
    device=AcceleratorDevice.AUTO,
)

In [ ]:
from docling.document_converter import DocumentConverter

source = "https://arxiv.org/pdf/2512.25075v1"  # document per local path or URL
converter = DocumentConverter()
result = converter.convert(source)

In [ ]:
from llm_pipeline.OllamaClient import OllamaEmbeddingClient
from docling.chunking import HybridChunker

embedder = OllamaEmbeddingClient()
chunker = HybridChunker(
            max_tokens=400,
            min_tokens=200,
        )  

document = getattr(result, "document", result)
chunks = list(chunker.chunk(document))

texts, metas = [], []

for idx, chunk in enumerate(chunks):
    text = getattr(chunk, "text", "").strip()
    if not text:
        continue

    section = None
    meta = getattr(chunk, "metadata", None)
    if isinstance(meta, dict):
        section = meta.get("section_path")
    metas.append(
        {
            "rp_abstract_id": "019b7bd0-9aaa-74dc-afb2-8bba8da42495",
            "chunk_index": idx,
            "section": section,
            "page_start": getattr(chunk, "page_start", None),
            "page_end": getattr(chunk, "page_end", None),
            "text": text,
            "doc_id" : "https://arxiv.org/pdf/2512.25075v1",
        }
    )
    texts.append(text)
embeddings = embedder.embed(texts)

records = []
for meta, vector in zip(metas, embeddings):
    meta["embedding"] = vector
    records.append(meta)


In [ ]:
for temp in chunks[1]:
    print(temp)

In [ ]:
meta = getattr(chunk, "meta", None)
print(meta.headings)



In [ ]:
page_start, page_end = extract_page_range(chunks[0])
print(page_start)
print(page_end)

In [ ]:
len(records)
# records[0]

In [ ]:
new_records = []
for temp in records:
    temp["doc_id"] = "https://arxiv.org/pdf/2512.25075v1"
    new_records.append(temp)

In [1]:
# from db.research_paper_data import insert_chunk_data
from services.ranker import rank_papers
from api.schemas import RankRequest, RankResponse, Paper

from configs.db_config import (
    POSTGRES_API_SERVER_POOL_SIZE,
    POSTGRES_API_SERVER_POOL_OVERFLOW,
    POSTGRES_API_SERVER_READ_ONLY_POOL_SIZE,
    POSTGRES_API_SERVER_READ_ONLY_POOL_OVERFLOW,

    )
from llm_pipeline.summary import LLMSummarizer
from db.engine.sql_engine import AsyncSqlEngine
from services.retrieval import get_intent_vector
import numpy as np


async def main():

    await AsyncSqlEngine.init_engine(
    pool_size=POSTGRES_API_SERVER_POOL_SIZE,
    max_overflow=POSTGRES_API_SERVER_POOL_OVERFLOW,
    app_name="main"
    )

    llmsummary = LLMSummarizer(batch_size=5, timeout=None)
    await llmsummary.run()
   

await main()

In [ ]:
# from db.research_paper_data import insert_chunk_data
from services.ranker import rank_papers
from api.schemas import RankRequest, RankResponse, Paper

from configs.db_config import (
    POSTGRES_API_SERVER_POOL_SIZE,
    POSTGRES_API_SERVER_POOL_OVERFLOW,
    POSTGRES_API_SERVER_READ_ONLY_POOL_SIZE,
    POSTGRES_API_SERVER_READ_ONLY_POOL_OVERFLOW,

    )
from db.engine.sql_engine import AsyncSqlEngine
from services.retrieval import get_intent_vector
import numpy as np

def normalize_intent_vectors(vectors: list[list[float]]) -> list[list[float]]:
    return [
        [float(x) for x in vec[0]]
        for vec in vectors
        if vec
    ]

async def main():

    await AsyncSqlEngine.init_engine(
    pool_size=POSTGRES_API_SERVER_POOL_SIZE,
    max_overflow=POSTGRES_API_SERVER_POOL_OVERFLOW,
    app_name="main"
    )

    intent_vec = await get_intent_vector(query= None, #"diffusion transformers",
                                         category=["cs.CV","cs.DC",])

    print(len(intent_vec))
    print(intent_vec)
    

    # for a in intent_vec:
    #     print(type(a))

    print(len(normalize_intent_vectors(intent_vec)))
    # await insert_chunk_data(new_records)
    rows = await rank_papers( intent_vectors=intent_vec,
                categories= ["cs.CV","cs.OS"],
                window_days= 30,
                top_k= 10,)
    
    return RankResponse(
            results=[
                Paper(
                    paper_id=str(r["id"]),
                    title=r["title"],
                    summary=r["summary"],
                    categories=r["primary_category"],
                    submitted_at=r["created_at"],
                    score=float(r["score"]),
                )
                for r in rows
            ]
    )
# asyncio.run(insert_chunk_data(records))

await main()

In [ ]:
result[0][0][0]

In [ ]:
intent_vec

In [ ]:
records[0]

In [ ]:
import time
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.threaded_standard_pdf_pipeline import ThreadedStandardPdfPipeline
from docling.utils.profiling import ProfilingItem


def document_reader(input_doc_path : str):

    pipeline_options = ThreadedPdfPipelineOptions(
        accelerator_options=AcceleratorOptions(
            device=AcceleratorDevice.CUDA,
        ),
        ocr_batch_size=4,
        layout_batch_size=64,
        table_batch_size=4,
    )
    # pipeline_options.do_ocr = False

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_cls=ThreadedStandardPdfPipeline,
                pipeline_options=pipeline_options,
            )
        }
    )

    start_time = time.time()
    doc_converter.initialize_pipeline(InputFormat.PDF)
    init_runtime = time.time() - start_time
    print(f"Pipeline initialized in {init_runtime:.2f} seconds.")

    start_time = time.time()
    conv_result = doc_converter.convert(input_doc_path)
    pipeline_runtime = time.time() - start_time
    assert conv_result.status == ConversionStatus.SUCCESS

    num_pages = len(conv_result.pages)
    print(f"Document converted in {pipeline_runtime:.2f} seconds.")
    print(f"  {num_pages / pipeline_runtime:.2f} pages/second.")

    return conv_result


# if __name__ == "__main__":
result = document_reader(input_doc_path= "https://arxiv.org/pdf/2512.25075v1")

In [ ]:
result.document.export_to_markdown

In [ ]:
# import logging
# import time
# from datetime import datetime, timedelta, timezone
# from pathlib import Path
# from typing import Any, Dict, List, Optional, Tuple

# import feedparser
# import pandas as pd
# import requests
# import yaml
# from pydantic import BaseModel, Field, ValidationError
# from pydantic_settings import BaseSettings
# from dateutil.relativedelta import relativedelta


# class ArxivSettings(BaseSettings):
#     # Core arXiv query config
#     category: str = Field(default="cs.*", description="arXiv category pattern")
#     max_results: int = Field(default=100, description="Max results per API page")
#     default_window: str = Field(default="1w", description="Default time window, e.g. 1d, 1w, 1m")
#     sort_by: str = Field(default="submittedDate", description="relevance | lastUpdatedDate | submittedDate")
#     sort_order: str = Field(default="descending", description="ascending | descending")
#     base_url: str = Field(default="https://export.arxiv.org/api/query", description="arXiv API base URL")

#     # HTTP / retry config
#     http_timeout_seconds: int = Field(default=10, description="HTTP request timeout")
#     http_max_retries: int = Field(default=3, description="Max HTTP retries")

#     # Logging
#     log_level: str = Field(default="INFO", description="Logging level")

#     # Optional path to YAML to pre-populate config
#     config_yaml_path: Optional[str] = Field(default=None, description="Optional YAML config path")

#     # class Config:
#     #     # env_prefix = "ARXIV_"  # e.g. ARXIV_CATEGORY, ARXIV_MAX_RESULTS
#     #     env_file = ".env"
#     #     extra = "ignore"


# def load_settings(config_yaml_path: Optional[str] = None) -> ArxivSettings:
#     """
#     Loads settings from optional YAML, then environment variables (env wins).

#     Order of precedence:
#         1. Environment variables / .env
#         2. YAML (if provided)
#         3. Defaults
#     """
#     yaml_data: Dict[str, Any] = {}
#     if config_yaml_path:
#         path = Path(config_yaml_path)
#         if path.is_file():
#             with path.open("r", encoding="utf-8") as f:
#                 yaml_data = yaml.safe_load(f) or {}

#     try:
#         settings = ArxivSettings(**yaml_data)
#     except ValidationError as e:
#         raise RuntimeError(f"Invalid configuration: {e}") from e

#     return settings


# def setup_logging(level_name: str) -> None:
#     level = getattr(logging, level_name.upper(), logging.INFO)
#     logging.basicConfig(
#         level=level,
#         format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
#     )


# class ArxivMetrics(BaseModel):
#     total_requests: int = 0
#     total_failures: int = 0
#     total_entries: int = 0
#     total_request_time_seconds: float = 0.0

#     def record_request(self, duration_seconds: float, success: bool, entries: int) -> None:
#         self.total_requests += 1
#         self.total_request_time_seconds += duration_seconds
#         if not success:
#             self.total_failures += 1
#         self.total_entries += entries

#     @property
#     def avg_request_latency(self) -> float:
#         if self.total_requests == 0:
#             return 0.0
#         return self.total_request_time_seconds / self.total_requests



# class ArxivClient:
#     def __init__(self, settings: ArxivSettings, metrics: Optional[ArxivMetrics] = None):
#         self.settings = settings
#         self.logger = logging.getLogger(self.__class__.__name__)
#         self.metrics = metrics or ArxivMetrics()

#     @staticmethod
#     def parse_window(window: str) -> Tuple[str, str]:
#         """
#         Parse a window string like "1d", "2w", "3m" into
#         (start_date, end_date) formatted as YYYYMMDD.
#         """
#         now = datetime.now(timezone.utc)
#         if len(window) < 2:
#             raise ValueError("Window must be like '1d', '2w', '3m' etc.")

#         num_str, unit = window[:-1], window[-1].lower()
#         if not num_str.isdigit():
#             raise ValueError(f"Invalid window numeric part: {window}")

#         value = int(num_str)

#         if unit == "d":
#             delta = timedelta(days=value)
#         elif unit == "w":
#             delta = timedelta(weeks=value)
#         elif unit == "m":
#             delta = relativedelta(months=value)
#         else:
#             raise ValueError(f"Invalid window unit '{unit}'. Use d, w, or m.")

#         start = now - delta
#         return start.strftime("%Y%m%d"), now.strftime("%Y%m%d")

#     def _http_request(self, params: Dict[str, Any]) -> str:
#         """
#         Low-level HTTP request with retries and logging.
#         """
#         last_exc: Optional[Exception] = None

#         for attempt in range(1, self.settings.http_max_retries + 1):
#             start_time = time.monotonic()
#             try:
#                 self.logger.debug("arXiv request attempt %d with params=%s", attempt, params)
#                 resp = requests.get(
#                     self.settings.base_url,
#                     params=params,
#                     timeout=self.settings.http_timeout_seconds,
#                 )
#                 duration = time.monotonic() - start_time

#                 if resp.status_code != 200:
#                     self.logger.warning(
#                         "Non-200 from arXiv (status=%s, attempt=%d)",
#                         resp.status_code,
#                         attempt,
#                     )
#                     self.metrics.record_request(duration, success=False, entries=0)
#                     last_exc = RuntimeError(f"HTTP {resp.status_code}")
#                 else:
#                     self.metrics.record_request(duration, success=True, entries=0)
#                     return resp.text

#             except Exception as exc:
#                 duration = time.monotonic() - start_time
#                 self.metrics.record_request(duration, success=False, entries=0)
#                 last_exc = exc
#                 self.logger.warning(
#                     "arXiv request failed (attempt=%d): %s",
#                     attempt,
#                     exc,
#                     exc_info=True,
#                 )

#             # simple backoff
#             time.sleep(min(2 ** attempt, 5))

#         raise RuntimeError(f"arXiv API request failed after {self.settings.http_max_retries} attempts") from last_exc

#     def fetch_entries(
#         self,
#         window: Optional[str] = None,
#         sort_by: Optional[str] = None,
#         sort_order: Optional[str] = None,
#     ) -> List[Any]:
#         """
#         Fetch all entries in the given window (relative time), paginating as needed.
#         """
#         window = window or self.settings.default_window
#         sort_by = sort_by or self.settings.sort_by
#         sort_order = sort_order or self.settings.sort_order

#         start_date, end_date = self.parse_window(window)
#         query = f"cat:{self.settings.category} AND submittedDate:[{start_date} TO {end_date}]"

#         self.logger.info(
#             "Fetching arXiv entries window=%s (%s -> %s), category=%s, sort_by=%s, sort_order=%s",
#             window,
#             start_date,
#             end_date,
#             self.settings.category,
#             sort_by,
#             sort_order,
#         )

#         all_entries: List[Any] = []
#         start_idx = 0

#         while True:
#             params = {
#                 "search_query": query,
#                 "start": start_idx,
#                 "max_results": self.settings.max_results,
#                 "sortBy": sort_by,
#                 "sortOrder": sort_order,
#             }

#             xml = self._http_request(params)
#             feed = feedparser.parse(xml)

#             entries = feed.entries or []
#             num_entries = len(entries)
#             self.metrics.total_entries += num_entries

#             self.logger.info(
#                 "Fetched page start=%d, got %d entries",
#                 start_idx,
#                 num_entries,
#             )

#             if num_entries == 0:
#                 break

#             all_entries.extend(entries)

#             # arXiv returns fewer than requested on the last page
#             if num_entries < self.settings.max_results:
#                 break

#             start_idx += self.settings.max_results

#         self.logger.info(
#             "Finished fetching. Total entries=%d, total_requests=%d, failures=%d, avg_latency=%.3fs",
#             len(all_entries),
#             self.metrics.total_requests,
#             self.metrics.total_failures,
#             self.metrics.avg_request_latency,
#         )
#         return all_entries

#     @staticmethod
#     def to_dataframe(entries: List[Any]) -> pd.DataFrame:
#         rows: List[Dict[str, Any]] = []

#         for e in entries:
#             # Authors
#             authors = []
#             for a in getattr(e, "authors", []):
#                 name = getattr(a, "name", None)
#                 if name:
#                     authors.append(name)

#             # PDF link
#             pdf_url = None
#             for link in getattr(e, "links", []):
#                 link_type = getattr(link, "type", None)
#                 href = getattr(link, "href", None)
#                 if link_type == "application/pdf":
#                     pdf_url = href
#                     break

#             primary_category = None
#             pc = getattr(e, "arxiv_primary_category", None)
#             if pc is not None:
#                 if isinstance(pc, dict):
#                     primary_category = pc.get("term")
#                 else:
#                     primary_category = getattr(pc, "term", None)

#             # All categories / tags
#             categories = []
#             for t in getattr(e, "tags", []):
#                 # t may be dict-like or obj with .term
#                 if isinstance(t, dict):
#                     term = t.get("term")
#                 else:
#                     term = getattr(t, "term", None)
#                 if term:
#                     categories.append(term)

#             row = {
#                 "id": getattr(e, "id", None),
#                 "title": getattr(e, "title", None),
#                 "summary": getattr(e, "summary", None),
#                 "authors": ", ".join(authors),
#                 "published": getattr(e, "published", None),
#                 "updated": getattr(e, "updated", None),
#                 "link": getattr(e, "link", None),
#                 "pdf_url": pdf_url,
#                 "primary_category": primary_category,
#                 "all_categories": ", ".join(categories),
#                 "doi": getattr(e, "arxiv_doi", None),
#                 "journal_ref": getattr(e, "arxiv_journal_ref", None),
#                 "comment": getattr(e, "arxiv_comment", None),
#             }

#             rows.append(row)

#         return pd.DataFrame(rows)


# def main():
#     raw_settings = ArxivSettings()
#     settings = load_settings(raw_settings.config_yaml_path)

#     setup_logging(settings.log_level)

#     logger = logging.getLogger("arxiv_main")
#     logger.info("Starting arXiv fetcher")

#     metrics = ArxivMetrics()
#     client = ArxivClient(settings=settings, metrics=metrics)

#     # You can also override window here (e.g., from CLI arg)
#     entries = client.fetch_entries(window=settings.default_window)
#     return entries

#     # df = client.to_dataframe(entries)
#     # logger.info("Created DataFrame with %d rows and %d columns", df.shape[0], df.shape[1])

#     # # Example: save to CSV for downstream processing
#     # output_path = f"arxiv_{settings.default_window}.csv"
#     # df.to_csv(output_path, index=False)
#     # logger.info("Saved results to %s", output_path)
